# Hyperparameter Tuning (Time-Aware, No Leakage)

This notebook isolates hyperparameter tuning and model selection.

Assumptions:
- You have already executed `feature_engineering.ipynb` to build causal, time-ordered features.
- You can provide the following artifacts in memory or by loading from disk: 
  - `combined_features_train` (DataFrame with `match_id`, `radiant_win`, and features)
  - `combined_features_test`  (same columns as train)
  - `train_data` (list of match objects in chronological order) or a sorted list of train match IDs: `train_ids_sorted`.

This notebook performs: 
- LightGBM hyperparameter search using a single forward validation split (last ~10% of train).
- Optional PCA dimensionality reduction (evaluated on the same split).
- Optional L1 feature selection via logistic regression.

To keep compute modest, we avoid k-fold CV and use early stopping.


In [ ]:
# Imports
import numpy as np
import pandas as pd
from typing import List, Dict, Any

from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel

import lightgbm as lgb


## Provide Inputs

Execute one of the following: 
- (Preferred) Assign `combined_features_train`, `combined_features_test`, and `train_data` from the feature engineering notebook (they should already be in the kernel).
- Or: Load previously saved CSV/Parquet files and provide `train_ids_sorted` (chronological train match IDs).


In [ ]:
# Example placeholders (uncomment and adapt if loading from disk):
# combined_features_train = pd.read_parquet('combined_features_train.parquet')
# combined_features_test = pd.read_parquet('combined_features_test.parquet')
# train_ids_sorted = list(pd.read_parquet('train_ids_sorted.parquet')['match_id'])

# If you ran feature_engineering.ipynb in this kernel, you likely have: 
# - combined_features_train, combined_features_test
# - train_data (list of matches in chronological order)
try:
    train_ids_sorted  # type: ignore[name-defined]
except NameError:
    # Derive sorted train IDs from train_data if available
    try:
        train_ids_sorted = [m.match_id for m in train_data]  # noqa: F821
    except Exception:
        train_ids_sorted = None

# Sanity check
assert 'combined_features_train' in globals(), 'Provide combined_features_train DataFrame'
assert 'combined_features_test' in globals(), 'Provide combined_features_test DataFrame'
if train_ids_sorted is None:
    print('WARNING: train_ids_sorted not provided; will fallback to a simple 90/10 split by row order (not time-aware).')


## Build Time-Aware Validation Split

We hold out the last 10% of the training time window as validation.


In [ ]:
df_train = combined_features_train.copy()
df_test  = combined_features_test.copy()
feature_cols = [c for c in df_train.columns if c not in ('match_id', 'radiant_win')]

if train_ids_sorted is not None:
    val_ratio = 0.10
    n_val = max(1, int(len(train_ids_sorted) * val_ratio))
    val_ids = set(train_ids_sorted[-n_val:])
    val_mask = df_train['match_id'].isin(val_ids)
else:
    # Fallback: simple row split (not time-aware)
    n = len(df_train)
    n_val = max(1, int(n * 0.10))
    val_mask = pd.Series([False]*(n-n_val) + [True]*n_val, index=df_train.index)

X_tr, y_tr = df_train.loc[~val_mask, feature_cols], df_train.loc[~val_mask, 'radiant_win'].astype(int)
X_val, y_val = df_train.loc[val_mask,  feature_cols], df_train.loc[val_mask,  'radiant_win'].astype(int)
X_te, y_te = df_test[feature_cols], df_test['radiant_win'].astype(int)

X_tr.shape, X_val.shape, X_te.shape


## LightGBM: Small, CPU-Friendly Search with Early Stopping

We try a tiny grid; early stopping halts training when validation does not improve.


In [ ]:
param_grid = [
    dict(num_leaves=31, min_data_in_leaf=200, feature_fraction=0.8, subsample=0.8, learning_rate=0.05),
    dict(num_leaves=63, min_data_in_leaf=300, feature_fraction=0.8, subsample=0.8, learning_rate=0.05),
    dict(num_leaves=63, min_data_in_leaf=500, feature_fraction=0.7, subsample=0.8, learning_rate=0.05),
    dict(num_leaves=127, min_data_in_leaf=800, feature_fraction=0.9, subsample=0.8, learning_rate=0.03),
]

best = None
for i, p in enumerate(param_grid, 1):
    model = lgb.LGBMClassifier(
        objective='binary',
        n_estimators=5000,
        early_stopping_rounds=100,
        verbosity=-1,
        random_state=42,
        n_jobs=-1,
        **p
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='binary_logloss',
        callbacks=[lgb.log_evaluation(period=100)]
    )
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    if (best is None) or (acc > best['acc']):
        best = dict(params=p, acc=acc, model=model)
    print(f'[{i}/{len(param_grid)}] val_acc={acc:.5f} params={p}')

print('Best val accuracy:', best['acc'])
print('Best params:', best['params'])


### Optional: Retrain Best Model and Score on Test


In [ ]:
best_params = best['params']
final_model = lgb.LGBMClassifier(
    objective='binary',
    n_estimators=best['model'].best_iteration_ or 1000,
    verbosity=-1,
    random_state=42,
    n_jobs=-1,
    **best_params
)
final_model.fit(df_train[feature_cols], df_train['radiant_win'].astype(int))
test_acc = accuracy_score(y_te, final_model.predict(X_te))
print('Test accuracy:', test_acc)


## PCA Exploration (Optional)

Evaluate PCA on the current feature set using the same validation split. Note: treat `match_id`/labels carefully and avoid leakage.


In [ ]:
def eval_pca(n_components_list: List[int]) -> pd.DataFrame:
    rows = []
    scaler = StandardScaler()
    Xtr_scaled = scaler.fit_transform(X_tr)
    Xval_scaled = scaler.transform(X_val)
    for n in n_components_list:
        pca = PCA(n_components=n, random_state=42)
        Xtr_p = pca.fit_transform(Xtr_scaled)
        Xval_p = pca.transform(Xval_scaled)
        model = lgb.LGBMClassifier(objective='binary', n_estimators=300, learning_rate=0.05, verbosity=-1, random_state=42)
        model.fit(Xtr_p, y_tr, eval_set=[(Xval_p, y_val)], eval_metric='binary_logloss', callbacks=[lgb.log_evaluation(period=100)])
        acc = accuracy_score(y_val, model.predict(Xval_p))
        rows.append({'n_components': n, 'val_acc': acc, 'explained_var': float(np.sum(pca.explained_variance_ratio_))})
    return pd.DataFrame(rows).sort_values('val_acc', ascending=False).reset_index(drop=True)

# Example: try a few sizes (adjust as desired)
# pca_results = eval_pca([8, 16, 25, 64])
# pca_results


## L1 Feature Selection (Optional)

Use L1-penalized logistic regression to select a subset of features; evaluate with LightGBM on the selected set.


In [ ]:
def eval_l1_selection(C_values: List[float]) -> pd.DataFrame:
    rows = []
    for C in C_values:
        selector_model = LogisticRegression(penalty='l1', solver='liblinear', C=C, random_state=42, max_iter=200)
        selector = SelectFromModel(selector_model)
        selector.fit(X_tr, y_tr)
        cols = np.array(feature_cols)[selector.get_support()]
        # Train LGBM on selected columns
        model = lgb.LGBMClassifier(objective='binary', n_estimators=600, learning_rate=0.05, verbosity=-1, random_state=42)
        model.fit(X_tr[cols], y_tr, eval_set=[(X_val[cols], y_val)], eval_metric='binary_logloss', callbacks=[lgb.log_evaluation(period=100)])
        acc = accuracy_score(y_val, model.predict(X_val[cols]))
        rows.append({'C': C, 'selected_features': int(len(cols)), 'val_acc': acc})
    return pd.DataFrame(rows).sort_values('val_acc', ascending=False).reset_index(drop=True)

# Example: try a few Cs (adjust as desired)
# l1_results = eval_l1_selection([0.001, 0.01, 0.1, 1.0])
# l1_results


# PCA

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator, TransformerMixin

def apply_pca_transformation(
    X_train_raw, 
    X_test_raw, 
    n_components, 
    random_state=42
):
    """
    Applies scaling and PCA to reduce the dimensionality of raw feature dataframes.

    This function correctly handles the fit/transform paradigm to prevent data
    leakage from the test set. It scales the data, applies PCA, and returns
    new dataframes with the principal components and the original 'match_id'.

    Args:
        X_train_raw (pd.DataFrame): The raw, high-dimensional training data. 
                                    Must include a 'match_id' column.
        X_test_raw (pd.DataFrame): The raw, high-dimensional testing data.
                                   Must include a 'match_id' column.
        n_components (int): The number of principal components to keep.
        random_state (int): The random state for PCA reproducibility.

    Returns:
        tuple: A tuple containing two pandas DataFrames:
               - X_train_pca_df (pd.DataFrame): Transformed training data with PCA features.
               - X_test_pca_df (pd.DataFrame): Transformed testing data with PCA features.
    """
    print(f"Applying PCA transformation with n_components={n_components}...")

    # --- 1. Input Validation ---
    if 'match_id' not in X_train_raw.columns or 'match_id' not in X_test_raw.columns:
        raise ValueError("Input DataFrames must contain a 'match_id' column.")

    # --- 2. Isolate Features and IDs ---
    train_ids = X_train_raw['match_id']
    test_ids = X_test_raw['match_id']
    
    X_train_features = X_train_raw.drop(columns=['match_id'])
    X_test_features = X_test_raw.drop(columns=['match_id'])

    # --- 3. Scaling (Fit on Train, Transform Both) ---
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_features)
    X_test_scaled = scaler.transform(X_test_features)

    # --- 4. PCA (Fit on Train, Transform Both) ---
    pca = PCA(n_components=n_components, random_state=random_state)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_features)
    
    # Report explained variance
    explained_variance = pca.explained_variance_ratio_.sum()
    print(f"Explained Variance: {explained_variance:.2%}")

    # --- 5. Create Result DataFrames ---
    pca_cols = [f'PC_{i+1}' for i in range(n_components)]
    
    X_train_pca_df = pd.DataFrame(X_train_pca, columns=pca_cols, index=X_train_raw.index)
    X_test_pca_df = pd.DataFrame(X_test_pca, columns=pca_cols, index=X_test_raw.index)

    # Re-attach match_id
    X_train_pca_df['match_id'] = train_ids
    X_test_pca_df['match_id'] = test_ids

    print("PCA transformation complete.")
    return X_train_pca_df, X_test_pca_df

In [ ]:
X_train_w2v_pca, X_test_w2v_pca = apply_pca_transformation(
    X_train_raw=w2v_features_train, 
    X_test_raw=w2v_features_test,
    n_components=8
)

Applying PCA transformation with n_components=8...
Explained Variance: 54.29%
PCA transformation complete.


/home/ubuntu/projects/dota2pred/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but PCA was fitted without feature names
  warnings.warn(


In [ ]:
run_experiment(
    feature_sets_train=[
        X_train_w2v_pca,
    ],
    feature_sets_test=[
        X_test_w2v_pca,
    ],
    y_test_df=test_outcome_df,
    y_train_df=train_outcome_df,
    models_dict=Models,
)

Starting new experiment run...


Training Logistic Regression...
Logistic Regression Accuracy: 0.534 (53.4%)

Training Random Forest...
Random Forest Accuracy: 0.529 (52.9%)

Training XGBoost...
XGBoost Accuracy: 0.524 (52.4%)

Training LightGBM...
LightGBM Accuracy: 0.534 (53.4%)

Experiment completed. Leaderboard:
                  name  accuracy
0  Logistic Regression  0.534485
1             LightGBM  0.533868
2        Random Forest  0.528686
3              XGBoost  0.524491
